<a href="https://colab.research.google.com/github/pandeynivedita7/codingal/blob/main/Voice_Assistant_(Offline)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# OfflineVoiceAssistant.py

import sounddevice as sd
import queue
import json
from vosk import Model, KaldiRecognizer
import pyttsx3
import datetime
import sys

# -------------------------------
# 1. CONFIGURATION
# -------------------------------

SAMPLE_RATE = 16000
CHANNELS = 1

# *** CHANGE THIS TO YOUR MODEL FOLDER ***
MODEL_PATH = r"C:\Users\arman\Downloads\Codingal\model"

# -------------------------------
# 2. LOAD MODEL (Correct Way)
# -------------------------------

try:
    model = Model(MODEL_PATH)                       # Must be Model(), not a string
except Exception as e:
    print("Model loading error:", e)
    sys.exit()

recognizer = KaldiRecognizer(model, SAMPLE_RATE)
audio_queue = queue.Queue()

tts_engine = pyttsx3.init()


# -------------------------------
# 3. RECORDING CALLBACK
# -------------------------------

def callback(indata, frames, time, status):
    if status:
        print("Audio Status:", status, file=sys.stderr)
    audio_queue.put(bytes(indata))                  # raw bytes into queue


# -------------------------------
# 4. SPEAK FUNCTION
# -------------------------------

def speak(text):
    tts_engine.say(text)
    tts_engine.runAndWait()


# -------------------------------
# 5. PROCESS COMMAND
# -------------------------------

def process_query(text):
    text = text.lower().strip()

    if "time" in text:
        now = datetime.datetime.now().strftime("%H:%M")
        return f"The time is {now}."

    if "date" in text:
        today = datetime.datetime.now().strftime("%B %d, %Y")
        return f"Today's date is {today}."

    if "hello" in text:
        return "Hello. How can I help you?"

    if "exit" in text or "stop" in text or "quit" in text:
        return "exit"

    return "I did not understand that."


# -------------------------------
# 6. MAIN LOOP
# -------------------------------

def main():
    print("Starting Voice Assistant…")
    print("Listening… Speak into your microphone.")

    try:
        with sd.RawInputStream(
            samplerate=SAMPLE_RATE,
            blocksize=8000,
            dtype='int16',
            channels=CHANNELS,
            callback=callback
        ):
            while True:
                data = audio_queue.get()

                if recognizer.AcceptWaveform(data):
                    result_json = recognizer.Result()
                    result = json.loads(result_json)

                    text = result.get("text", "")

                    if text:
                        print("You said:", text)

                        response = process_query(text)

                        if response == "exit":
                            print("Stopping program…")
                            speak("Goodbye.")
                            break

                        print("Assistant:", response)
                        speak(response)

                else:
                    # Optional: show live partial results
                    partial = json.loads(recognizer.PartialResult()).get("partial", "")
                    if partial:
                        print("\rListening: " + partial, end="")

    except KeyboardInterrupt:
        print("\nProgram interrupted.")

    except Exception as e:
        print("Error:", e)


# -------------------------------
# 7. START PROGRAM
# -------------------------------

if __name__ == "__main__":
    main()


Step-by-step explanation

Install dependencies & get a model

pip install sounddevice vosk pyttsx3

Download a Vosk speech model (e.g. English small model) and extract it to a folder named model in the same directory as the script. The Vosk models are distributed as folders containing files like am, conf, etc.

Constants and initialization

SAMPLE_RATE = 16000 — Vosk models usually expect 16000 Hz. Use the model's sample rate.

Load the Vosk Model(MODEL_PATH) and create KaldiRecognizer(model, SAMPLE_RATE).

Initialize pyttsx3 TTS engine to speak responses.

Audio callback

callback(indata, frames, time, status) is called by sounddevice whenever a chunk of audio is available.

We convert indata to bytes and push it into a thread-safe queue.Queue() for processing in the main loop. This keeps the callback fast and non-blocking.

Processing recognition results

The main loop reads raw chunks from audio_queue.

For each chunk, recognizer.AcceptWaveform(data) returns True when a final chunk forms a recognizable utterance. Then recognizer.Result() returns a JSON string with the recognized text.

recognizer.PartialResult() returns the partial transcription while the speaker is still talking (optional; we print it).

Handling user intent

process_query() takes recognized text and decides an appropriate response (time, date, greeting, exit, or fallback).

If the user says "stop"/"exit"/"quit", process_query returns the special token "exit" that the main loop interprets to stop the program.

Text-to-speech

speak() uses pyttsx3 to speak responses synchronously (runAndWait() blocks until speech finishes).

Running and stopping

Run the script. It opens the microphone stream and listens. Use Ctrl+C to terminate or say "exit"/"stop" to let the assistant stop gracefully.

Troubleshooting & tips

If you hear nothing or recognition quality is poor:

Ensure microphone is selected correctly (sounddevice may use system default). On platforms with multiple devices, set device= in RawInputStream.

Check microphone gain and reduce background noise.

If you get an error loading the model, verify MODEL_PATH points to a valid Vosk model folder.

On Windows you may need to install a wheel for sounddevice dependencies (PortAudio). Install via pip normally handles it; otherwise search for platform-specific instructions.

To support multiple languages or commands, expand process_query() and/or use an NLP library for intents.